# Drug Dataset — NLP Text Cleaning Pipeline

This notebook cleans the `drugs_data.parquet` dataset through a structured NLP pipeline:

1. Load & inspect
2. Strip section header boilerplate
3. Normalize casing
4. Remove special characters & Unicode artifacts
5. Deduplicate repeated paragraphs within cells
6. Handle high-missingness columns
7. Normalize the `route` column
8. Deduplicate brand+generic pairs
9. Export cleaned dataset

## 0. Install Dependencies

In [ ]:
# Install required packages (run once)
!pip install pyarrow pandas --quiet

## 1. Load & Inspect

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_parquet('drugs_data.parquet')

print(f'Shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nDtypes:\n{df.dtypes}')

In [ ]:
# Null counts and percentages
null_summary = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct':   (df.isnull().mean() * 100).round(1)
})
print('Null Summary:')
print(null_summary.to_string())

In [ ]:
# Preview first 3 rows
df.head(3)

## 2. Strip Section Header Boilerplate

Each text column contains embedded section labels (`"INDICATIONS: ..."`, `"4 CONTRAINDICATIONS ..."`, `"DOSAGE Adults-"`) that are noise for downstream NLP tasks.

In [ ]:
# Map each column to the section header keywords likely to appear in it
HEADER_PATTERNS = {
    'indications':       r'^[\d\s]*\b(INDICATIONS?|USES?|USE|HOMEOPATHIC USES?)\b[\s:\-]*',
    'dosage':            r'^[\d\s]*\b(DOSAGE(\s+AND\s+ADMINISTRATION)?|DIRECTIONS?)\b[\s:\-]*',
    'contraindications': r'^[\d\s]*\b(CONTRAINDICATIONS?)\b[\s:\-]*',
    'side_effects':      r'^[\d\s]*\b(ADVERSE\s+REACTIONS?|SIDE\s+EFFECTS?)\b[\s:\-]*',
    'warnings':          r'^[\d\s]*\b(WARNINGS?(\s+AND\s+PRECAUTIONS?)?)\b[\s:\-]*',
    'do_not_use':        r'^[\d\s]*\b(DO\s+NOT\s+USE|CONTRAINDICATIONS?)\b[\s:\-]*',
    'use_when':          r'^[\d\s]*\b(WHEN\s+USING|USE\s+WHEN)\b[\s:\-]*',
}

def strip_section_header(text, pattern):
    if pd.isna(text):
        return text
    cleaned = re.sub(pattern, '', str(text), flags=re.IGNORECASE).strip()
    # Also strip a leading colon/dash that may remain after the header
    cleaned = re.sub(r'^[:\-–—]\s*', '', cleaned)
    return cleaned if cleaned else text  # fallback to original if result is empty

df_clean = df.copy()

for col, pattern in HEADER_PATTERNS.items():
    if col in df_clean.columns:
        before = df_clean[col].dropna().str[:40].tolist()[:2]
        df_clean[col] = df_clean[col].apply(lambda x: strip_section_header(x, pattern))
        after  = df_clean[col].dropna().str[:40].tolist()[:2]
        print(f'[{col}]')
        print(f'  Before: {before}')
        print(f'  After:  {after}\n')

## 3. Normalize Casing

`brand_name` and `generic_name` are inconsistently cased — some ALL CAPS, some Title Case. Standardize to Title Case.

In [ ]:
print('Before:')
print(df_clean[['brand_name', 'generic_name']].head(6).to_string())

df_clean['brand_name']   = df_clean['brand_name'].str.strip().str.title()
df_clean['generic_name'] = df_clean['generic_name'].str.strip().str.title()
df_clean['route']        = df_clean['route'].str.strip().str.upper()

print('\nAfter:')
print(df_clean[['brand_name', 'generic_name']].head(6).to_string())

## 4. Remove Special Characters & Unicode Artifacts

Issues found:
- **510** rows with zero-width spaces (`\u200b`) in `warnings`
- **891** rows with markdown bold markers (`**...**`) in `indications`
- **3,979** rows with `[see Warnings and Precautions (5.1)]`-style cross-reference anchors in `side_effects`
- Non-breaking spaces (`\u00a0`)

In [ ]:
def clean_special_chars(text):
    if pd.isna(text):
        return text
    t = str(text)
    # Unicode artifacts
    t = t.replace('\u200b', ' ')          # zero-width space
    t = t.replace('\u00a0', ' ')          # non-breaking space
    t = t.replace('\u2019', "'")          # right single quotation mark
    t = t.replace('\u2018', "'")          # left single quotation mark
    t = t.replace('\u201c', '"')          # left double quotation mark
    t = t.replace('\u201d', '"')          # right double quotation mark
    t = t.replace('\u2013', '-')          # en dash
    t = t.replace('\u2014', '-')          # em dash
    # Markdown bold/italic markers
    t = re.sub(r'\*{1,3}', '', t)
    # Cross-reference anchors like [see Warnings and Precautions (5.1)]
    t = re.sub(r'\[see [^\]]+\]', '', t, flags=re.IGNORECASE)
    # Numbered section references like (5.1), (6.1)
    t = re.sub(r'\(\s*\d+\.\d+\s*\)', '', t)
    # Extra whitespace
    t = re.sub(r'[ \t]{2,}', ' ', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    return t.strip()

TEXT_COLS = ['indications', 'dosage', 'contraindications', 'side_effects',
             'warnings', 'do_not_use', 'use_when']

for col in TEXT_COLS:
    df_clean[col] = df_clean[col].apply(clean_special_chars)

print('Special character cleaning applied to all text columns.')

# Verify zero-width spaces are gone
remaining_zws = df_clean['warnings'].astype(str).str.contains('\u200b').sum()
print(f'Remaining zero-width spaces in warnings: {remaining_zws}')

## 5. Deduplicate Repeated Paragraphs Within Cells

`side_effects` and `warnings` fields contain entire paragraphs repeated verbatim — a copy-paste artifact from FDA label sources. This is removed by sentence-level deduplication.

In [ ]:
def dedup_paragraphs(text):
    """Remove duplicate sentences/paragraphs within a single text field."""
    if pd.isna(text):
        return text
    t = str(text)
    # Split on sentence-ending punctuation or double newlines
    segments = re.split(r'(?<=[.!?])\s+|\n{2,}', t)
    seen = set()
    unique_segments = []
    for seg in segments:
        key = re.sub(r'\s+', ' ', seg.strip().lower())
        if key and key not in seen:
            seen.add(key)
            unique_segments.append(seg.strip())
    return ' '.join(unique_segments)

# Apply to columns known to have repeated content
for col in ['side_effects', 'warnings', 'contraindications']:
    before_len = df_clean[col].dropna().str.len().mean()
    df_clean[col] = df_clean[col].apply(dedup_paragraphs)
    after_len  = df_clean[col].dropna().str.len().mean()
    reduction  = (1 - after_len / before_len) * 100
    print(f'[{col}] Avg length: {before_len:.0f} → {after_len:.0f} chars ({reduction:.1f}% reduction)')

## 6. Handle High-Missingness Columns

Several columns are majority null. Strategy:
- Merge `do_not_use` into `warnings` (they are semantically related)
- Merge `use_when` into `indications` (context for use)
- Fill remaining nulls with `"Not specified"` so rows are preserved

In [ ]:
# ── Merge do_not_use → warnings ────────────────────────────────────────────
def merge_fields(base, extra, separator=' | DO NOT USE: '):
    if pd.isna(extra) or str(extra).strip() == '':
        return base
    if pd.isna(base) or str(base).strip() == '':
        return str(extra)
    return str(base) + separator + str(extra)

df_clean['warnings'] = df_clean.apply(
    lambda r: merge_fields(r['warnings'], r['do_not_use']), axis=1
)

# ── Merge use_when → indications ──────────────────────────────────────────
df_clean['indications'] = df_clean.apply(
    lambda r: merge_fields(r['indications'], r['use_when'], separator=' | USE WHEN: '), axis=1
)

# Drop the now-merged source columns
df_clean = df_clean.drop(columns=['do_not_use', 'use_when'])
print('Dropped: do_not_use, use_when (merged into warnings and indications)')

# ── Add a presence indicator for sparse columns ────────────────────────────
df_clean['has_contraindications'] = df_clean['contraindications'].notna().astype(int)
df_clean['has_side_effects']      = df_clean['side_effects'].notna().astype(int)
print('Added binary indicator columns: has_contraindications, has_side_effects')

# ── Fill remaining nulls ──────────────────────────────────────────────────
fill_cols = ['indications', 'dosage', 'contraindications', 'side_effects', 'warnings']
df_clean[fill_cols] = df_clean[fill_cols].fillna('Not specified')

print('\nNull counts after handling:')
print(df_clean.isnull().sum())

## 7. Normalize the `route` Column

Route values are inconsistent. Map them to a controlled vocabulary.

In [ ]:
print('Raw route value counts:')
print(df_clean['route'].value_counts().head(20))

In [ ]:
ROUTE_MAP = {
    'ORAL':                        'oral',
    'TOPICAL':                     'topical',
    'OPHTHALMIC':                  'ophthalmic',
    'INTRAVENOUS':                 'intravenous',
    'SUBCUTANEOUS':                'subcutaneous',
    'INTRAMUSCULAR':               'intramuscular',
    'NASAL':                       'nasal',
    'RECTAL':                      'rectal',
    'VAGINAL':                     'vaginal',
    'TRANSDERMAL':                 'transdermal',
    'INHALATION':                  'inhalation',
    'AURICULAR (OTIC)':            'otic',
    'DENTAL':                      'dental',
    'INTRATHECAL':                 'intrathecal',
    'INTRAARTICULAR':              'intraarticular',
    'SUBLINGUAL':                  'sublingual',
    'BUCCAL':                      'buccal',
}

df_clean['route'] = (
    df_clean['route']
    .str.strip()
    .str.upper()
    .map(ROUTE_MAP)
    .fillna('unknown')
)

print('Normalized route value counts:')
print(df_clean['route'].value_counts())

## 8. Deduplicate brand+generic Pairs

**608** rows share the same `brand_name` + `generic_name`. Keep the most informative row (the one with the most non-null / non-empty fields).

In [ ]:
print(f'Rows before deduplication: {len(df_clean)}')
print(f'Duplicate brand+generic pairs: {df_clean.duplicated(subset=["brand_name","generic_name"]).sum()}')

# Score each row by number of non-null, non-empty, non-"Not specified" fields
text_quality_cols = ['indications', 'dosage', 'contraindications', 'side_effects', 'warnings']

df_clean['_quality_score'] = df_clean[text_quality_cols].apply(
    lambda row: sum(
        1 for v in row
        if pd.notna(v) and str(v).strip() not in ('', 'Not specified')
    ),
    axis=1
)

# Keep the highest-quality row per brand+generic pair
df_clean = (
    df_clean
    .sort_values('_quality_score', ascending=False)
    .drop_duplicates(subset=['brand_name', 'generic_name'], keep='first')
    .drop(columns=['_quality_score'])
    .reset_index(drop=True)
)

print(f'\nRows after deduplication: {len(df_clean)}')

## 9. Final Inspection

In [ ]:
print('=== Final Dataset Summary ===')
print(f'Shape: {df_clean.shape}')
print(f'\nColumns: {df_clean.columns.tolist()}')
print(f'\nNull counts:\n{df_clean.isnull().sum()}')
print(f'\nRoute distribution:\n{df_clean["route"].value_counts()}')

In [ ]:
# Quick before/after text comparison for a sample row
idx = 0
print('=== Sample Row — indications ===')
print('ORIGINAL:', repr(df['indications'].iloc[idx]))
print()
print('CLEANED: ', repr(df_clean['indications'].iloc[idx]))

In [ ]:
# Text length distributions after cleaning
print('Avg text lengths (cleaned):')
for col in ['indications', 'dosage', 'warnings', 'side_effects', 'contraindications']:
    lengths = df_clean[col].astype(str).str.len()
    print(f'  {col:20s}: mean={lengths.mean():.0f}, median={lengths.median():.0f}, max={lengths.max()}')

## 10. Export Cleaned Dataset

In [ ]:
output_path = 'drugs_data_cleaned.parquet'
df_clean.to_parquet(output_path, index=False)
print(f'Saved cleaned dataset → {output_path}')
print(f'Final shape: {df_clean.shape}')

In [ ]:
# Verify the saved file loads correctly
df_verify = pd.read_parquet(output_path)
print(f'Verified shape: {df_verify.shape}')
df_verify.head(3)